# 05 · Apêndice: resultados depois do imposto de renda
Simula o imposto de renda do investidor pessoa física sobre as carteiras do 03. É um apêndice descritivo: não muda nenhuma regra, carteira ou resultado do 01 a 04, e nenhum número daqui serve para escolher parâmetro.

**Regra simulada** (`docs/pesquisa/tributacao.csv`): 15% sobre o ganho líquido do mês em operações comuns; ganho de cada venda pelo custo médio ponderado de compra; prejuízo compensa ganhos do mês e dos meses seguintes, sem prazo; isenção se as vendas do mês forem de até R$ 20 mil.

## 1. Parâmetros e carga
Lê as carteiras do `backtest.pkl`. Tabelas em `results/tabelas` com prefixo `05_` e figuras em `results/figuras/imposto/`, separadas das do 04.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PASTA = Path('../data/processed')
PASTA_TABELAS = Path('../results/tabelas')
PASTA_IMPOSTO = Path('../results/figuras/imposto')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
PASTA_IMPOSTO.mkdir(parents=True, exist_ok=True)

ALIQUOTA = 0.15
LIMITE_ISENCAO = 20000.0
CAPITAL_COM_ISENCAO = 100000.0
CUSTO_BPS = 20.0
RETORNO_AUSENTE = 0.0
FATIAS = {'treino': ('2020-09-30', '2021-12-31'),
          'validacao': ('2022-01-31', '2023-12-31'),
          'teste': ('2024-01-31', '2025-12-31')}
FIGURAS_FATIAS = ['validacao', 'teste']
VERSOES = {'iMOM-BR': 'E_residual_filtro', 'Residual sem filtro': 'D_residual', 'Bruto com filtro': 'C_bruto_filtro', 'Bruto sem filtro': 'B_bruto', 'Só filtro': 'A_filtro'}
F = 'iMOM-BR'

bt = pd.read_pickle(PASTA / 'backtest.pkl')
retorno = pd.read_pickle(PASTA / 'retorno_mensal.pkl')
ultimo_com_preco = pd.read_pickle(PASTA / 'saidas.pkl')['ultimo_mes_com_preco']
datas = retorno.index
refs = bt['referencias']
CORTE_SAIDA = bt['parametros']['corte_saida']
assert CUSTO_BPS == bt['parametros']['custo_bps']
print(len(refs), 'meses | alíquota', ALIQUOTA, '| custo', CUSTO_BPS, 'bps por lado')

## 2. Simulação com imposto
A cada mês: rebalanceia para os pesos do 03, apura o ganho das vendas pelo custo médio, deduz o custo da operação, compensa prejuízos e paga o imposto. Custo e imposto saem das posições proporcionalmente, reduzindo também o custo de compra. Com alíquota zero, a simulação tem que reproduzir a série do 03 líquida de custo: a conferência trava o notebook se não reproduzir.

In [ ]:
def retorno_no_mes(codigos, s):
    anterior = datas[datas.get_loc(s) - 1]
    valores = retorno.loc[s, codigos].copy()
    faltou = valores.isna()
    saiu = faltou & (ultimo_com_preco.reindex(codigos) == anterior).to_numpy()
    valores[saiu] = CORTE_SAIDA
    valores[faltou & ~saiu] = RETORNO_AUSENTE
    return valores


def simular(chave, meses, aliquota=ALIQUOTA, com_isencao=False, capital=CAPITAL_COM_ISENCAO):
    pesos = bt['pesos'][chave]
    posicao, custo_compra = {}, {}
    total, prejuizo = capital, 0.0
    linhas = {}
    for s in meses:
        alvos = pesos.loc[s][pesos.loc[s] > 0] * total
        ganho = vendas = compras = 0.0
        for codigo in list(posicao):
            alvo = alvos.get(codigo, 0.0)
            if posicao[codigo] > alvo:
                venda = posicao[codigo] - alvo
                fracao = venda / posicao[codigo]
                ganho += venda - custo_compra[codigo] * fracao
                custo_compra[codigo] *= 1 - fracao
                posicao[codigo] = alvo
                vendas += venda
                if alvo == 0.0:
                    del posicao[codigo], custo_compra[codigo]
        for codigo, alvo in alvos.items():
            atual = posicao.get(codigo, 0.0)
            if atual < alvo:
                compra = alvo - atual
                posicao[codigo] = alvo
                custo_compra[codigo] = custo_compra.get(codigo, 0.0) + compra
                compras += compra
        custo_operacao = CUSTO_BPS / 10000 * (vendas + compras)
        resultado = ganho - custo_operacao
        isento = com_isencao and vendas <= LIMITE_ISENCAO
        if isento:
            imposto = 0.0
            prejuizo += min(resultado, 0.0)
        else:
            base = resultado + prejuizo
            imposto = aliquota * max(base, 0.0)
            prejuizo = min(base, 0.0)
        valores = retorno_no_mes(list(posicao), s)
        posicao = {k: v * (1 + valores[k]) for k, v in posicao.items()}
        anterior, antes_de_pagar = total, sum(posicao.values())
        total = antes_de_pagar - custo_operacao - imposto
        fator = total / antes_de_pagar
        posicao = {k: v * fator for k, v in posicao.items()}
        custo_compra = {k: v * fator for k, v in custo_compra.items()}
        ganho_latente = sum(posicao.values()) - sum(custo_compra.values()) + prejuizo
        linhas[s] = {'retorno': total / anterior - 1, 'valor': total, 'imposto': imposto, 'vendas': vendas, 'isento': isento,
                     'valor_se_vender_tudo': total - aliquota * max(ganho_latente, 0.0)}
    return pd.DataFrame(linhas).T


todos_os_meses = bt['series']['E_residual_filtro'].index
for nome, chave in VERSOES.items():
    tabela = bt['series'][chave]
    esperado = tabela['retorno_bruto'] - CUSTO_BPS / 10000 * tabela['giro']
    obtido = simular(chave, todos_os_meses, aliquota=0.0)['retorno'].astype(float)
    assert (obtido - esperado).abs().max() < 1e-10, nome
print('com alíquota zero, a simulação reproduz as cinco séries do 03 líquidas de custo: ok')

## 3. Resultados por fatia
Cada fatia começa do zero, comprando a carteira inteira no primeiro mês. Por isso a coluna "antes do imposto" pode diferir um pouco da série líquida do 04, que já vinha investida. O cenário principal não usa a isenção, e o resultado em % não depende do capital. O cenário com isenção usa uma carteira de R$ 100 mil e confere a regra **mês a mês**: só não paga imposto o mês em que o total vendido no próprio mês fica em até R$ 20 mil (a coluna `meses_isentos` conta esses meses); o mês em que as vendas passam de R$ 20 mil paga 15% sobre o ganho daquele mês, compensando prejuízos anteriores. Ibovespa e CDI aparecem brutos. O Sharpe é sempre contra o CDI bruto, como no 04.

In [ ]:
def medidas(r, cdi):
    x = r - cdi
    curva = (1 + r).cumprod()
    return {'retorno_acumulado_%': (curva.iloc[-1] - 1) * 100, 'retorno_anual_%': (curva.iloc[-1] ** (12 / len(r)) - 1) * 100,
            'volatilidade_anual_%': r.std(ddof=1) * np.sqrt(12) * 100, 'sharpe': x.mean() / x.std(ddof=1) * np.sqrt(12) if x.std(ddof=1) > 0 else np.nan,
            'queda_maxima_%': (curva / np.maximum(curva.cummax(), 1) - 1).min() * 100}


CENARIOS = {'antes do imposto': {'aliquota': 0.0}, 'pós-imposto sem isenção': {}, 'pós-imposto com isenção': {'com_isencao': True}}
SIMULADO = {}


def rodar_imposto(fatia):
    meses = pd.date_range(*FATIAS[fatia], freq='ME')
    cdi = refs['CDI'].loc[meses]
    linhas = {}
    for nome, chave in VERSOES.items():
        for cenario, argumentos in CENARIOS.items():
            s = simular(chave, meses, **argumentos)
            SIMULADO[(fatia, nome, cenario)] = s
            r = s['retorno'].astype(float)
            linhas[(nome, cenario)] = {**medidas(r, cdi), 'imposto_pago_%_do_capital': s['imposto'].sum() / CAPITAL_COM_ISENCAO * 100,
                                       'retorno_se_vender_tudo_no_fim_%': (s['valor_se_vender_tudo'].iloc[-1] / CAPITAL_COM_ISENCAO - 1) * 100,
                                       'meses_isentos': int(s['isento'].astype(bool).sum())}
    for nome in ['Ibovespa', 'CDI']:
        linhas[(nome, 'bruto')] = medidas(refs[nome].loc[meses], cdi)
    tabela = pd.DataFrame(linhas).T.rename_axis(['serie', 'cenario']).astype(float)
    print(f'=== fatia {fatia}: {len(meses)} meses ===')
    print(tabela.round(2).to_string())
    tabela.to_csv(PASTA_TABELAS / f'05_{fatia}_pos_imposto.csv')
    return tabela


TABELAS = {fatia: rodar_imposto(fatia) for fatia in FATIAS}

## 4. Figuras
Só da validação e do teste, em `results/figuras/imposto/`: curva do iMOM-BR antes e depois do imposto contra Ibovespa e CDI, retorno de cada versão antes e depois do imposto, e imposto pago por mês no iMOM-BR.

In [ ]:
ROXO_ESCURO, LILAS, CINZA, CINZA_CLARO = '#230042', '#A88BD6', '#6E6A78', '#A8A3B3'
CORES = {'antes': LILAS, 'depois': ROXO_ESCURO, 'Ibovespa': CINZA, 'CDI': CINZA_CLARO}
plt.rcParams.update({'figure.figsize': (9, 4.8), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})


def salvar(fatia, numero, nome):
    plt.tight_layout()
    plt.savefig(PASTA_IMPOSTO / f'05_{fatia}_{numero:02d}_{nome}.png', dpi=150)
    plt.show()
    plt.close('all')


def figuras_imposto(fatia):
    meses = pd.date_range(*FATIAS[fatia], freq='ME')
    inicio = meses[0] - pd.offsets.MonthEnd(1)

    def curva(r):
        c = ((1 + r.astype(float)).cumprod() - 1) * 100
        c.loc[inicio] = 0.0
        return c.sort_index()

    antes = SIMULADO[(fatia, F, 'antes do imposto')]['retorno']
    depois = SIMULADO[(fatia, F, 'pós-imposto sem isenção')]['retorno']
    fig, ax = plt.subplots()
    ax.plot(curva(antes).index, curva(antes), color=CORES['antes'], lw=2, label='iMOM-BR antes do imposto (líquida de 20 bps)')
    ax.plot(curva(depois).index, curva(depois), color=CORES['depois'], lw=2.4, label='iMOM-BR pós-imposto')
    ax.plot(curva(refs['Ibovespa'].loc[meses]).index, curva(refs['Ibovespa'].loc[meses]), color=CORES['Ibovespa'], lw=1.4, label='Ibovespa (bruto)')
    ax.plot(curva(refs['CDI'].loc[meses]).index, curva(refs['CDI'].loc[meses]), color=CORES['CDI'], lw=1.4, ls='--', label='CDI (bruto)')
    ax.axhline(0, color='k', lw=0.8)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    ax.set_title(f'iMOM-BR antes e depois do imposto: retorno acumulado (%) ({fatia})')
    ax.set_ylabel('% desde o início da fatia')
    ax.legend(frameon=False, ncol=2)
    salvar(fatia, 1, 'curva_pos_imposto')

    t = TABELAS[fatia]['retorno_acumulado_%']
    nomes = list(VERSOES)
    x = np.arange(len(nomes))
    fig, ax = plt.subplots()
    ax.bar(x - 0.2, [t[(n, 'antes do imposto')] for n in nomes], width=0.38, color=CORES['antes'], label='antes do imposto')
    ax.bar(x + 0.2, [t[(n, 'pós-imposto sem isenção')] for n in nomes], width=0.38, color=CORES['depois'], label='pós-imposto')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x, nomes, rotation=15)
    ax.grid(axis='x', visible=False)
    ax.set_title(f'Retorno acumulado por versão, líquido de 20 bps (%) ({fatia})')
    ax.set_ylim(top=t.loc[nomes].max() * 1.18)
    ax.legend(frameon=False, ncol=2, loc='upper center')
    salvar(fatia, 2, 'retorno_por_versao')

    imposto = SIMULADO[(fatia, F, 'pós-imposto sem isenção')]['imposto'].astype(float) / CAPITAL_COM_ISENCAO * 100
    fig, ax = plt.subplots()
    ax.bar(imposto.index, imposto, width=20, color=CORES['depois'])
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    ax.grid(axis='x', visible=False)
    ax.set_title(f'iMOM-BR: imposto pago por mês, % do capital inicial ({fatia})')
    ax.set_ylabel('% do capital inicial')
    salvar(fatia, 3, 'imposto_por_mes')


for fatia in FIGURAS_FATIAS:
    figuras_imposto(fatia)

## 5. A isenção mês a mês: R$ 100 mil aplicados uma vez
O investidor aplica R$ 100 mil no primeiro mês da fatia e não aplica mais nada. A cada mês a carteira do iMOM-BR é rebalanceada, e a isenção é conferida **no próprio mês**: se o total vendido no mês ficar em até R$ 20 mil, aquele mês não paga imposto, mesmo que a soma das vendas de vários meses passe muito de R$ 20 mil. No mês em que as vendas passam do limite, paga 15% sobre o ganho líquido do mês, compensando prejuízos anteriores. A tabela compara, em reais, o mesmo investidor sem imposto, pagando imposto todo mês e usando a isenção. É descritivo: não muda nenhuma regra nem resultado.

In [ ]:
CENARIOS_REAIS = ['antes do imposto', 'pós-imposto sem isenção', 'pós-imposto com isenção']


def resumo_em_reais(s):
    patrimonio = float(s['valor'].iloc[-1])
    return {'meses': len(s), 'aplicado_R$': CAPITAL_COM_ISENCAO, 'patrimonio_final_R$': patrimonio, 'lucro_R$': patrimonio - CAPITAL_COM_ISENCAO,
            'lucro_%': (patrimonio / CAPITAL_COM_ISENCAO - 1) * 100, 'imposto_pago_R$': float(s['imposto'].sum()),
            'meses_isentos': int(s['isento'].astype(bool).sum()), 'meses_com_venda_acima_de_20_mil': int((s['vendas'].astype(float) > LIMITE_ISENCAO).sum()),
            'maior_venda_num_mes_R$': float(s['vendas'].max()), 'patrimonio_se_vender_tudo_no_fim_R$': float(s['valor_se_vender_tudo'].iloc[-1])}


resumo_reais = pd.DataFrame({(fatia, cenario): resumo_em_reais(SIMULADO[(fatia, F, cenario)]) for fatia in FATIAS for cenario in CENARIOS_REAIS}).T.rename_axis(['fatia', 'cenario'])
resumo_reais.to_csv(PASTA_TABELAS / '05_isencao_resumo.csv')
pd.set_option('display.float_format', lambda v: f'{v:,.0f}')
print(resumo_reais.to_string())
pd.reset_option('display.float_format')

for fatia in FATIAS:
    m = SIMULADO[(fatia, F, 'pós-imposto com isenção')][['valor', 'vendas', 'isento', 'imposto']].astype({'valor': float, 'vendas': float, 'imposto': float})
    m['isento'] = m['isento'].astype(bool)
    m.rename_axis('mes').round(2).to_csv(PASTA_TABELAS / f'05_{fatia}_isencao_mes_a_mes.csv')
    if fatia == 'teste':
        print()
        print('teste, mês a mês (R$):')
        print(m.round(0).to_string())

for fatia in FIGURAS_FATIAS:
    m = SIMULADO[(fatia, F, 'pós-imposto com isenção')].astype({'valor': float, 'vendas': float, 'imposto': float})
    sem = SIMULADO[(fatia, F, 'pós-imposto sem isenção')]['valor'].astype(float)
    antes = SIMULADO[(fatia, F, 'antes do imposto')]['valor'].astype(float)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [1.4, 1]})
    ax1.axhline(CAPITAL_COM_ISENCAO / 1000, color=CINZA_CLARO, lw=1.2, ls='--', label='aplicado: R$ 100 mil')
    ax1.plot(antes.index, antes / 1000, color=LILAS, lw=3.2, label='antes do imposto')
    ax1.plot(sem.index, sem / 1000, color=CINZA, lw=1.6, ls='--', label='pós-imposto sem isenção')
    ax1.plot(m.index, m['valor'] / 1000, color=ROXO_ESCURO, lw=1.8, label='pós-imposto com isenção mês a mês')
    ax1.set_ylabel('R$ mil'); ax1.legend(frameon=False, fontsize=8)
    ax1.set_title(f'iMOM-BR com R$ 100 mil aplicados uma vez: patrimônio ({fatia})')
    cores = [LILAS if v <= LIMITE_ISENCAO else ROXO_ESCURO for v in m['vendas']]
    ax2.bar(m.index, m['vendas'] / 1000, width=20, color=cores)
    ax2.axhline(LIMITE_ISENCAO / 1000, color='black', lw=1, ls='--')
    ax2.text(m.index[0], LIMITE_ISENCAO / 1000, ' limite de R$ 20 mil no mês', va='bottom', fontsize=8)
    ax2.set_ylabel('vendas no mês (R$ mil)')
    ax2.set_title('Vendas de cada mês: lilás = até R$ 20 mil (isento), roxo = acima (paga 15% sobre o ganho líquido do mês)', fontsize=9)
    ax2.set_xlim(m.index[0] - pd.Timedelta(days=20), m.index[-1] + pd.Timedelta(days=20))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
    fig.tight_layout(); fig.savefig(PASTA_IMPOSTO / f'05_{fatia}_04_isencao_mes_a_mes.png', dpi=200); plt.show()

## Resumo e limitações
**O que o 05 entrega:** uma tabela `05_<fatia>_pos_imposto.csv` por fatia e 3 figuras por fatia (validação e teste) em `results/figuras/imposto/`; mais a isenção mês a mês em reais para R$ 100 mil aplicados uma vez (`05_isencao_resumo.csv`, `05_<fatia>_isencao_mes_a_mes.csv` e a figura `05_<fatia>_04_isencao_mes_a_mes.png`).

**Limitações:** o preço ajustado embute proventos no ganho, e dividendos eram isentos, o que pode superestimar o imposto; a retenção de 15% sobre o JCP não está modelada, o que pode subestimar; a retenção de 0,005% na fonte não está modelada, porque compensa o imposto do mês; o imposto é pago no próprio mês, e não no mês seguinte; a venda para pagar custo e imposto não gera ganho adicional; Ibovespa e CDI aparecem brutos, embora também sejam tributados, então a comparação é desfavorável ao iMOM-BR; a regra precisa ser conferida com a Receita ou um contador antes de citada no paper.